<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 08 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Accelerate Queries Without Changing the Answer</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:920px;margin:0">Reuse a purchase report, precompute repeated aggregation and enrichment, then investigate one product with an inverted index. Verify the answer before interpreting the plan and runtime evidence.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">View · Synchronous materialized view · Asynchronous materialized view · Query rewrite · Inverted Index · Query Profile</span>
</div>

The commerce team has two recurring needs: a revenue dashboard and a product-level investigation. Work through six steps:

1. Check the source data and prepare the event and product tables.
2. Demonstrate a regular view: compare answers, selected scans, and actual scan work.
3. Use a synchronous materialized view to keep summaries consistent with committed writes.
4. Use an asynchronous materialized view to store and refresh a multi-table category report.
5. Use transparent query rewrite to reuse the asynchronous materialized view, then test its semantic boundary.
6. Isolate an inverted index for selective event lookup.

**Run from top to bottom.** The reset cell recreates only the explicitly named `m08_*` objects used here. To repeat the experiment or recover after a partial run, start again at initialization and run all cells. Do not run two copies of this lab concurrently. Other modules' tables remain available.

### Initialize the Lab

Run the next cell before Section 1. It loads the shared course helper and creates the `lab` object used by every later cell. Run it again after restarting the Jupyter kernel. It does not start Docker or change data in Doris.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Prepare the event and product tables

Use the existing local single-node Doris 4.1.3 environment from Lab 1, with these tables in `doris_course`:

| Required table | Prepared in | Used here for |
|---|---|---|
| `events_modelled` | Lab 4 | One day of typed event detail |
| `dim_products` | Lab 6 | Product categories and brands |

The event scope is **[2020-03-01 00:00:00, 2020-03-02 00:00:00)**: the start is included and the end is excluded. The course dataset contains **75,259 events** in this interval.

The next cell checks this event count and confirms that the product table is nonempty. If either check fails, complete the corresponding earlier lab before continuing. No external data service or download is needed.

In [2]:
from decimal import Decimal
from doris_course.ui import card

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

events_ready = lab.sql("""
SELECT COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-01' AND event_time < '2020-03-02'
""", title="Events in [2020-03-01, 2020-03-02)")
assert int(events_ready.iloc[0]["event_count"]) == 75_259, "Expected 75,259 events. Check the Lab 4 data."

products_ready = lab.sql("SELECT COUNT(*) AS product_count FROM dim_products",
                         title="Available products")
assert int(products_ready.iloc[0]["product_count"]) > 0, "dim_products is empty. Complete Lab 6 first."


event_count
75259


product_count
204231


**Expected result:** `event_count` is **75,259** for the specified interval, and `product_count` is greater than zero. These checks confirm the expected event quantity and the availability of product data; they do not verify every field value.

### Prepare the lab tables

The report counts purchase events and sums `revenue` by region. Its grain is one row per region; it does not count distinct customers.

The following cell resets only this lab's objects and copies events from the checked interval. `m08_events` preserves event detail with the Duplicate Key model. Its Key columns determine sort order, not uniqueness. `m08_products` retains one current row per product with the Unique Key model. Only products referenced by the copied events are needed.

In [3]:
lab.wait_for_mv_jobs("m08_events", "m08_sync_metrics", "m08_async_category_metrics")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS m08_async_category_metrics")
lab.execute("DROP VIEW IF EXISTS m08_purchase_view")
# Dropping m08_events also removes its own synchronous materialized index.
for table in ("m08_events_indexed", "m08_events_unindexed", "m08_events", "m08_products"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")

lab.execute("""
CREATE TABLE m08_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO m08_events
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM events_modelled
WHERE event_time >= '2020-03-01' AND event_time < '2020-03-02'
""", title="Copy one day of event detail")
lab.execute("""
CREATE TABLE m08_products (
    product_id BIGINT NOT NULL,
    category VARCHAR(32),
    brand VARCHAR(32)
)
UNIQUE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO m08_products
SELECT product_id, category, brand
FROM dim_products
WHERE product_id IN (SELECT product_id FROM m08_events)
""", title="Copy the referenced products")
lab.execute("ANALYZE TABLE m08_events WITH SYNC")
lab.execute("ANALYZE TABLE m08_products WITH SYNC")
lab.sql("""
SELECT COUNT(*) AS unmatched_purchase_events
FROM m08_events e LEFT ANTI JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
""", title="Purchases without a matching product");

unmatched_purchase_events
501


**Expected result:** `unmatched_purchase_events` is **501** with the unchanged course data. These are 501 purchase events for product **`1005115`**, not 501 different products. Lab 6 deliberately excluded this product when creating `dim_products` (`WHERE product_id <> 1005115`) to demonstrate an unmatched relationship. Because `m08_products` is copied from that dimension, the product remains absent here; this is not a copy failure.

The regional report includes these events because it reads the event table directly. The category report later uses an Inner Join with the product table and excludes them, so its totals need not match the regional report.

`ANALYZE ... WITH SYNC` collects optimizer statistics for these owned tables; it does not create a materialized result.

## 2. Reuse purchase logic with a regular view

This step demonstrates a regular view: how it reuses query logic and what work remains when the report runs. Several reports need the same rule: include only purchase events. A regular view saves that rule so each report does not have to repeat `WHERE event_type = 'purchase'`.

We will answer three questions separately: do the answers match, which table does each plan read, and how many rows did each scan process?

To observe the scans and computation actually performed by each query, the lab temporarily disables three caches that can reuse previous results or filtering information: SQL Cache, Query Cache, and Condition Cache. This avoids attributing cache benefits to a view or an index. The original session settings are restored after each capture, including when capture fails.

The helper also enables Profile level 2 to collect execution details. Storage, operating-system, and inverted-index caches remain in place, so elapsed time is not a cold-cache benchmark.

**A. Compare the answers.** Create the ordinary view, then run the regional report directly against `m08_events` and through `m08_purchase_view`. The view already contains the purchase filter, so `view_query` does not repeat it.

Read the two small result tables below: each row is one region, `event_count` counts purchase events, and `total_revenue` sums their revenue. The check compares every group and measure.

In [ ]:
lab.execute("DROP VIEW IF EXISTS m08_purchase_view")
lab.execute("""
CREATE VIEW m08_purchase_view AS
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM m08_events
WHERE event_type = 'purchase'
""")
regional_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m08_events
WHERE event_type = 'purchase'
GROUP BY region
ORDER BY region
"""
view_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m08_purchase_view
GROUP BY region
ORDER BY region
"""
baseline = lab.capture_query(regional_query)
regular_view = lab.capture_query(view_query)
baseline.show_result("Direct query: purchases by region")
regular_view.show_result("Through the view: purchases by region")
lab.assert_same_rows(baseline.rows, regular_view.rows)
card("All 8 regional groups have the same purchase counts and revenue.",
     "ok", "Answers match");

**Expected result:** both tables contain the same eight regional groups. For example, `region_01` has 2,319 purchase events and revenue of 732,977.08 with the unchanged course data. This verifies that using the view preserves the report's answer. It does not yet tell us whether the database performed less work.

**B. Check what each plan reads.** The following cell displays only the selected `TABLE:` lines from the captured `EXPLAIN` plans. In `m08_events(m08_events)`, the name before the parentheses is the table, and the name inside is its selected base index. Complete plans are available in the collapsed panels.

This cell displays evidence already captured in part A; it does not run the reports again.

In [5]:
baseline.show_scan_plan("Direct query: selected scan")
regular_view.show_scan_plan("Through the view: selected scan")
lab.assert_scan(baseline, "m08_events")
lab.assert_scan(regular_view, "m08_events");

**Expected result:** both selected scan lines name `m08_events(m08_events)`. Although the second SQL names the regular view, its plan still reads the underlying event table. The ordinary view stores a query definition rather than a separate persisted report result. The plan shows the access path; it does not measure actual scanned rows.

**C. Compare the actual scan work.** Read `ScanRows` and `RowsProduced` under each `OLAP_SCAN_OPERATOR` below. These are raw counters from the merged Query Profile, with the Fragment, Pipeline, and table/index identity retained. For this comparison, use the `sum` value; `avg`, `max`, and `min` describe the contributing execution instances and must not be added to it.

- `ScanRows`: rows read by this scan from storage, including page-cache reads.
- `RowsProduced`: rows this scan sends to later operators after scan-side filtering. This is not the final report row count.

The complete raw Profiles remain available in collapsed panels for further inspection.

In [6]:
baseline.show_scan_rows("Direct query: actual scan rows")
regular_view.show_scan_rows("Through the view: actual scan rows");

**Expected result:** on the tested dataset, both scans report `ScanRows: sum 75.259K (75259)` and `RowsProduced: sum 18.298K (18298)`:

```text
75,259 event rows scanned
    → 18,298 purchase events emitted by the scan
    → 8 regional groups produced by the later aggregation
```

The regular view reused the filter definition, but did not reduce the recorded scan work in these executions. Equal query results, the selected base-table scans, and the runtime row counts support three separate observations. They do not establish a fixed timing relationship.

The next section keeps the same report SQL and tests whether a synchronous materialized view can supply precomputed aggregates instead.

## 3. Keep summaries current with a synchronous materialized view

The regular view reused a filter definition, but both queries still scanned event detail. The dashboard repeatedly needs the same counts and revenue, so now precompute those measures with a **synchronous materialized view**.

We will check four things separately: the initial build finishes, the report answer stays the same, the plan selects the materialized index, and the actual scan work decreases. Afterward, we will append a purchase to test write consistency.

### A. Create the summary and wait for the initial build

The definition groups events by `(region, event_type)` and stores an event count and revenue sum for each group. Keeping `event_type` lets the purchase report exclude other event types. Keeping `region` lets it produce the same regional totals as before. For example, purchases and cart events in `region_01` belong to different groups.

The `mv_` aliases give the stored columns names distinct from the base-table columns, as required by the tested build. The view is created on the owned Duplicate Key event table; this example does not imply that every query or Table Model supports the same definition.

Initial construction runs as a background job. `sync_before` records older Job IDs so the wait checks this new build rather than an earlier success. Read **Build state** in the compact output; the full task record is available in a collapsed panel.

`ANALYZE TABLE ... WITH SYNC` then collects optimizer statistics for this table. Here, `WITH SYNC` means waiting for statistics collection to finish; it is separate from the materialized view's write-maintenance behavior.

In [ ]:
lab.wait_for_mv_jobs("m08_events", "m08_sync_metrics", "m08_async_category_metrics")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS m08_sync_metrics ON m08_events")
sync_before = {str(row["JobId"]) for row in lab.sync_mv_jobs("m08_events", "m08_sync_metrics")}
lab.execute("""
CREATE MATERIALIZED VIEW m08_sync_metrics AS
SELECT region AS mv_region,
       event_type AS mv_event_type,
       COUNT(*) AS mv_event_count,
       SUM(revenue) AS mv_total_revenue
FROM m08_events
GROUP BY region, event_type
""")
lab.wait_for_sync_mv("m08_events", "m08_sync_metrics", sync_before)
lab.execute("ANALYZE TABLE m08_events WITH SYNC");

**Expected result:** the output names base table `m08_events`, materialized view `m08_sync_metrics`, and build state **`FINISHED`**. The initial structure is ready. This does not yet prove that a report query will select it. The helper raises an error if the build fails or does not finish within its bounded wait.

### B. Run the original report and check its answer

Run the same `regional_query` used in Step 2. It still names `m08_events` and filters purchase events; we do not change it to select from the synchronous view.

`capture_query` executes the report once and saves its result, plan, and runtime Profile. This cell displays only the eight-row result. `assert_same_rows` compares every region and both measures against the saved pre-materialization baseline; it does not compare only grand totals.

In [8]:
sync_report = lab.capture_query(regional_query)
sync_report.show_result("Regional purchases after synchronous materialization")
lab.assert_same_rows(baseline.rows, sync_report.rows)
card("Every region has the same purchase count and revenue as the Step 2 baseline.",
     "ok", "Report answer unchanged");

region,event_count,total_revenue
region_01,2319,732977.08
region_02,2199,713831.18
region_03,2270,679830.92
region_04,2201,704615.19
region_05,2323,675808.09
region_06,2399,748752.94
region_07,2243,674899.85
region_08,2344,739526.04


**Expected result:** the report still has eight regional groups. For example, `region_01` still has 2,319 purchase events and revenue of 732,977.08. The check confirms that the answer is unchanged. Result equality alone does not tell us which storage structure supplied it or how much work was performed.

### C. Confirm that the plan selected the materialized index

Compare the selected scan lines before and after creation. The name before parentheses is the base table; the name inside parentheses is the selected index. Full plans are folded below each excerpt.

These are the plans already captured with the two report executions. Displaying them does not rerun the queries.

In [9]:
baseline.show_scan_plan("Before creation: selected scan")
sync_report.show_scan_plan("After creation: selected scan")
lab.assert_scan(sync_report, "m08_events", "m08_sync_metrics");

**Expected result:** the selected access path changes from:

```text
m08_events(m08_events)
```

to:

```text
m08_events(m08_sync_metrics)
```

The SQL still references the base table, but the new plan selects the synchronous materialized index. This is how this lab uses the synchronous view; it is not queried as a separate table. This proves plan selection, not the number of rows actually scanned. If the expected scan is absent, inspect the complete plan before claiming that the view was used.

### D. Compare the actual scan work recorded in Query Profile

`baseline` and `sync_report` were created earlier by `lab.capture_query()`. That helper enabled profiling, executed each query, and saved its Query Profile together with its result and plan. The calls below do not rerun the SQL: `show_profile_scan_rows()` reads each saved Profile's `MergedProfile` and extracts only **`ScanRows`** and **`RowsProduced`** under the named scan operator.

`ScanRows` records rows read by the scan, including page-cache reads. `RowsProduced` records rows emitted by that scan to later operators. Neither is automatically the number of rows in the final report. The table/index labels tie each runtime counter to the access path established by `EXPLAIN` in Section C; complete raw Profiles remain folded for optional inspection.


In [ ]:
baseline.show_profile_scan_rows("Before creation: base-table scan work")
sync_report.show_profile_scan_rows("After creation: materialized-index scan work");

**Expected result:** the baseline records 75,259 scanned event rows and emits 18,298 purchase events. In the verified run, the materialized-index scan reads 64 stored summary rows and emits 32 purchase-summary rows. Both report executions still return the same eight regional groups.

```text
Base-table path:
75,259 event rows scanned → 18,298 purchase events → 8 regional results

Observed materialized-index path:
64 summary rows scanned → 32 purchase-summary rows → 8 regional results
```

The two paths read rows at different grains: raw events versus precomputed summaries. Partial groups can be stored in multiple Tablets and must be combined by later aggregation, so scanning 64 summary rows does not mean the report should have 64 rows. Exact summary-row counts can vary with storage state; use the counters shown in your run rather than requiring a fixed value of 64 or 32.

Together, the checks show that this execution selected the materialized index, returned the same answer, and scanned fewer rows. They do not establish a fixed latency speedup or measure the extra storage and write-maintenance cost. The next experiment checks whether the answer also stays correct after a committed write.

### E. Verify synchronous maintenance after one committed purchase

Append one purchase worth `19.95` in `region_01`, then run the unchanged regional report without issuing a refresh. The earlier checks established the complete baseline; this focused check displays and verifies only the group affected by the new row and the selected scan.


In [11]:
lab.insert("""
INSERT INTO m08_events
SELECT CAST('2020-03-01 12:00:00' AS DATETIME), -8001, -8001,
       'purchase', 'region_01',
       (
           SELECT MIN(p.product_id)
           FROM m08_products p
           JOIN m08_events e ON p.product_id = e.product_id
           WHERE e.event_type = 'purchase'
       ),
       CAST(19.95 AS DECIMAL(12,2))
WHERE NOT EXISTS (SELECT 1 FROM m08_events WHERE event_id = -8001)
""", title="Append one owned purchase once")


1

The complete report SQL is visible below. `verify_query_change()` executes the `SELECT` and verifies the `region_01` changes. The separate `EXPLAIN` call makes the plan evidence explicit and displays only its selected `TABLE` line.


In [ ]:
after_insert_sql = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m08_events
WHERE event_type = 'purchase'
GROUP BY region
ORDER BY region
"""

after_insert_rows = lab.verify_query_change(
    after_insert_sql,
    baseline.rows,
    match={"region": "region_01"},
    expected_changes={"event_count": 1, "total_revenue": Decimal("19.95")},
    title="Observed change in region_01",
)
lab.explain_selected_scans(
    "EXPLAIN " + after_insert_sql,
    title="After the write: selected scan",
    expected_table="m08_events",
    expected_index="m08_sync_metrics",
);


**Expected result:**

- The result excerpt shows `event_count_change = 1` and `total_revenue_change = 19.95` for `region_01`. These values prove that the committed purchase is present in the report result.
- The cropped `EXPLAIN` excerpt shows `TABLE: doris_course.m08_events(m08_sync_metrics)`. This proves that the plan selected the synchronous materialized index.

No manual refresh occurs between the insert and the report. The result excerpt and plan excerpt answer different questions: the result verifies the updated answer, while `EXPLAIN` identifies the planned access path. The full report, full plan, and Query Profile are not displayed because they add no new evidence to this focused check.


## 4. Store and refresh a multi-table report with an asynchronous materialized view

The next report groups purchases by category and region. It requires a Join with the current product attributes, which the single-table summary cannot supply. An asynchronous materialized view stores this multi-table result and can be queried directly.

Use `BUILD DEFERRED` to separate definition from the first refresh, `REFRESH COMPLETE ON MANUAL` to make the refresh contract explicit, and `grace_period = 0` to avoid opting into a stale-result grace period for transparent rewrite. Directly reading the stored view and asking the optimizer to rewrite a base-table query are distinct operations.

In [ ]:
lab.wait_for_mv_jobs("m08_events", "m08_sync_metrics", "m08_async_category_metrics")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS m08_async_category_metrics")

category_query = """
SELECT p.category, e.region,
       COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category, e.region
ORDER BY p.category, e.region
"""
stored_category_query = """
SELECT category, region, event_count, total_revenue
FROM m08_async_category_metrics
ORDER BY category, region
"""
lab.execute("""
CREATE MATERIALIZED VIEW m08_async_category_metrics
BUILD DEFERRED REFRESH COMPLETE ON MANUAL
DISTRIBUTED BY HASH(category) BUCKETS 4
PROPERTIES ("replication_num" = "1", "grace_period" = "0")
AS
SELECT p.category, e.region,
       COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category, e.region
""")
refresh_before = {str(row["TaskId"]) for row in lab.async_mv_tasks("m08_async_category_metrics")}
lab.execute("REFRESH MATERIALIZED VIEW m08_async_category_metrics COMPLETE")
lab.wait_for_async_refresh("m08_async_category_metrics", refresh_before)
lab.sql("""
SELECT Name, State, RefreshState, RefreshInfo, SyncWithBaseTables
FROM mv_infos('database'='doris_course')
WHERE Name = 'm08_async_category_metrics'
""", title="Asynchronous view status")
with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    category_before, stored_before = lab.compare_queries(
        category_query, stored_category_query,
        left_label="source tables", right_label="stored view",
        title="First refresh: complete result comparison",
    )


**How to read the output:**

1. **Completed manual refresh task:** the new Task ID reaches `SUCCESS`, uses `COMPLETE` refresh, and reports 100% progress. The helper waits for the newly submitted task, so an older successful task cannot satisfy this check.
2. **Asynchronous view status:** the explicit SQL projection shows only the object name, object state, latest refresh state, declared refresh policy, and whether the stored result is synchronized with its base tables.
3. **Complete result comparison:** both SQL statements return 64 category/region groups and `differing_rows = 0`. The helper compares every column and duplicate multiplicity while avoiding two 64-row displays. Materialized-view rewrite is disabled for the source query, so this does not accidentally compare the stored view with itself.

The Notebook still passes the visible `category_query` and `stored_category_query` SQL strings into the comparison helper. The shortened table is only a presentation summary; the equality check still uses the complete query results.


### Change one product's category and inspect the stale stored result

Choose a product that has purchase events, then move it to `lab8_reclassified` in the owned dimension. This changes which category should receive its existing purchases without changing the overall event count or revenue.

Read the stored view before requesting another refresh. Also display the base-table plan with normal rewrite enabled; interpret the plan you actually see rather than assuming that direct stale reads and transparent rewrite behave alike.

In [ ]:
chosen_product = lab.sql("""
SELECT p.product_id, p.category, COUNT(*) AS purchase_events
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.product_id, p.category
ORDER BY purchase_events DESC, p.product_id
LIMIT 1
""", title="Product to reclassify")
changed_product_id = int(chosen_product.iloc[0]["product_id"])
original_category = str(chosen_product.iloc[0]["category"])
assert original_category != "lab8_reclassified", "Restart from Section 1 to repeat this experiment."
lab.execute(f"""
UPDATE m08_products SET category = 'lab8_reclassified'
WHERE product_id = {changed_product_id}
""")
with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    category_changed, stored_stale = lab.summarize_query_values(
        category_query, stored_category_query,
        left_label="current source", right_label="stored view before refresh",
        value_column="category",
        values=(original_category, "lab8_reclassified"),
        metrics=("event_count", "total_revenue"),
        title="Old and new category before refresh",
    )
lab.assert_same_rows(stored_before, stored_stale)
assert "lab8_reclassified" in set(category_changed["category"])
assert "lab8_reclassified" not in set(stored_stale["category"])
assert category_changed["event_count"].sum() == category_before["event_count"].sum()
assert category_changed["total_revenue"].sum() == category_before["total_revenue"].sum()
with lab.session_settings({"enable_materialized_view_rewrite": "true"}):
    lab.explain_selected_scans("EXPLAIN " + category_query,
                               title="Selected scans while the stored view is stale")
card("The stored view is unchanged, while the source answer moves purchases into lab8_reclassified.",
     "ok", "Refresh lag observed");

**How to read the output:**

Compare the same category across the two `result` rows:

- In the **current source**, 508 purchases and `122846.80` revenue have moved out of `category_8` and into `lab8_reclassified`.
- In the **stored view before refresh**, `category_8` still contains those purchases, while `lab8_reclassified` has zero groups, events, and revenue.
- The silent assertions also confirm that the overall event count and revenue did not change. This is a reclassification, not an insertion or deletion.
- The cropped `EXPLAIN` lists the source-table scans selected while the stored view is stale. It describes the plan; it neither refreshes the view nor measures runtime work.

The helper receives the complete `category_query` and `stored_category_query` SQL strings. It executes both full results but displays only the original and new categories needed to understand the stale snapshot.


In [ ]:
refresh_before = {str(row["TaskId"]) for row in lab.async_mv_tasks("m08_async_category_metrics")}
lab.execute("REFRESH MATERIALIZED VIEW m08_async_category_metrics COMPLETE")
lab.wait_for_async_refresh("m08_async_category_metrics", refresh_before)
with lab.session_settings({"enable_materialized_view_rewrite": "false"}):
    category_after_refresh, stored_fresh = lab.summarize_query_values(
        category_query, stored_category_query,
        left_label="current source", right_label="refreshed stored view",
        value_column="category",
        values=(original_category, "lab8_reclassified"),
        metrics=("event_count", "total_revenue"),
        title="Old and new category after refresh",
    )
lab.assert_same_rows(category_changed, category_after_refresh)
lab.assert_same_rows(category_after_refresh, stored_fresh)
lab.execute("ANALYZE TABLE m08_products WITH SYNC")
lab.execute("ANALYZE TABLE m08_async_category_metrics WITH SYNC")
card("The refreshed category/region groups now match the changed source answer.",
     "ok", "Refresh reconciliation passed");

**Expected result:**

- The new refresh Task ID reaches `SUCCESS`, uses `COMPLETE` refresh, and reports 100% progress.
- For both the current source and refreshed stored view, `category_8` now contains 1,814 events and `551991.97` revenue, while `lab8_reclassified` contains 508 events and `122846.80` revenue.

The assertions still compare every column and duplicate multiplicity across all 72 groups. Rewrite is disabled for the source query so that the verification cannot compare the materialized view with itself. The full refresh recomputes the stored query; this lab does not measure its production resource cost.


## 5. Reuse an asynchronous materialized view through transparent query rewrite

The application can keep querying the base tables; it does not need to name `m08_async_category_metrics`. With transparent rewrite enabled, Doris may substitute the refreshed view when that view contains enough information to derive the requested answer.

For this view, the eligible query below:

- uses the same `m08_events`–`m08_products` Join and the same `event_type = 'purchase'` scope;
- groups by `category`, which Doris can obtain by rolling up the view's finer `category, region` groups; and
- requests only `COUNT(*)` and `SUM(revenue)`, the additive measures preserved by the view.

The view must also have a usable refreshed result. These conditions describe the rewrite verified in this lab; the optimizer still decides whether a particular query is eligible. Compare the **same base-table SQL** with rewrite disabled and enabled. Both executions return the same answer, but their Query Profiles should identify different runtime scans: the source tables when rewrite is disabled and `m08_async_category_metrics` when it is enabled.

In [ ]:
eligible_query = """
SELECT p.category, COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category
ORDER BY p.category
"""
without_rewrite = lab.capture_query(eligible_query,
    settings={"enable_materialized_view_rewrite": "false"})
without_rewrite.show_result("Rewrite disabled: category totals")
without_rewrite.show_profile_scan_rows("Rewrite disabled: source-table scan work")
with_rewrite = lab.capture_query(eligible_query,
    settings={"enable_materialized_view_rewrite": "true"})
with_rewrite.show_result("Rewrite enabled: category totals")
with_rewrite.show_profile_scan_rows("Rewrite enabled: materialized-view scan work")
lab.assert_same_rows(without_rewrite.rows, with_rewrite.rows)
lab.assert_scan(without_rewrite, "m08_events")
lab.assert_scan(without_rewrite, "m08_products")
lab.assert_scan(with_rewrite, "m08_async_category_metrics")
card("Complete category totals match; the enabled plan selects the asynchronous view.",
     "ok", "Eligible rewrite checked");


**How to read the output:**

- **Rewrite disabled: category totals** is the answer computed from the source tables.
- **Rewrite enabled: category totals** is the answer returned when Doris may reuse the asynchronous materialized view. The two displayed tables should contain the same nine rows, and the complete row-set assertion checks every value.
- With rewrite disabled, the Query Profile identifies runtime scans of `m08_events(m08_events)` and `m08_products(m08_products)`.
- With rewrite enabled, the Query Profile identifies `m08_async_category_metrics(m08_async_category_metrics)`.
- `ScanRows` and `RowsProduced` are actual execution counters for those scans. Complete raw Profiles remain folded for optional inspection.

No `EXPLAIN` output is displayed in this code box. Plan expectations are still checked silently by `assert_scan()`. The counters describe this verified run; they are not a general speedup ratio.


### Ask for a measure the view did not preserve

A base-table query cannot use this view merely because it has the same Join, filter, and grouping dimensions. The requested measures must also be derivable from the state stored in the view.

Now ask for distinct buyers per category. `m08_async_category_metrics` preserves event counts and revenue sums, but it does not preserve `user_id` values or a mergeable distinct-buyer state. Those stored measures cannot reconstruct how many different buyers produced the events. Even a distinct count stored separately for each region would not generally be additive across regions, because one buyer can appear in more than one region.

For this query, Doris cannot derive the answer from `m08_async_category_metrics`. Enabling rewrite therefore leaves the source-table plan in place, and the Query Profile still identifies runtime scans of `m08_events(m08_events)` and `m08_products(m08_products)`.

In [ ]:
ineligible_query = """
SELECT p.category, COUNT(DISTINCT e.user_id) AS distinct_buyers
FROM m08_events e JOIN m08_products p ON e.product_id = p.product_id
WHERE e.event_type = 'purchase'
GROUP BY p.category
ORDER BY p.category
"""
ineligible_baseline = lab.capture_query(ineligible_query,
    settings={"enable_materialized_view_rewrite": "false"})
ineligible_baseline.show_result("Rewrite disabled: distinct buyers")
ineligible_baseline.show_profile_scan_rows("Rewrite disabled: source-table scan work")
ineligible = lab.capture_query(ineligible_query,
    settings={"enable_materialized_view_rewrite": "true"})
ineligible.show_result("Rewrite enabled: distinct buyers")
ineligible.show_profile_scan_rows("Rewrite enabled: source-table scan work")
lab.assert_same_rows(ineligible_baseline.rows, ineligible.rows)
lab.assert_scan(ineligible, "m08_events")
lab.assert_scan(ineligible, "m08_products")
card("Distinct-buyer results match; the selected scans still read the source tables.",
     "ok", "Missing-measure boundary checked");


**How to read the output:**

- The rewrite-disabled and rewrite-enabled result tables should contain the same nine distinct-buyer totals. The assertion checks every returned value.
- Both Query Profiles identify `m08_events(m08_events)` and `m08_products(m08_products)`. The rewrite-enabled execution does not identify `m08_async_category_metrics`.
- The materialized view stores event counts and revenue sums, but it does not preserve `user_id` or a mergeable distinct-buyer state, so Doris must read source detail.

No `EXPLAIN` output is displayed in this code box. Plan expectations are still checked silently by `assert_scan()`. This is a limitation of this view definition, not evidence that every distinct-count query is ineligible for materialized-view acceleration.


## 6. Isolate an inverted index for product investigation

A support analyst wants the events for one rare product, rather than a precomputed aggregate. Create two event tables with identical columns, Duplicate Key order, distribution, and input rows. Only one declares an Inverted Index on `product_id`.

`product_id` is numeric, so this is an equality index with no text tokenizer. It is neither a full-text `MATCH` example nor a Join Runtime Filter. The predicate does not constrain the leading sort column or the `user_id` distribution column. Declaring the index before loading the data avoids a separate historical index-build step.

In [ ]:
for table in ("m08_events_indexed", "m08_events_unindexed"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")

lab.execute("""
CREATE TABLE m08_events_unindexed (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
CREATE TABLE m08_events_indexed (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    INDEX m08_product_idx(product_id) USING INVERTED
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")
for table in ("m08_events_unindexed", "m08_events_indexed"):
    lab.insert(f"INSERT INTO {table} SELECT * FROM m08_events", title=f"Load {table}")
    lab.execute(f"ANALYZE TABLE {table} WITH SYNC")

rare_product = lab.sql("""
SELECT product_id, COUNT(*) AS matching_events
FROM m08_events_unindexed
GROUP BY product_id
HAVING SUM(revenue) > 0
ORDER BY matching_events, product_id
LIMIT 1
""", title="Product selected for the indexed lookup")
rare_product_id = int(rare_product.iloc[0]["product_id"]);

**Expected result:** both tables are loaded from the same current `m08_events` snapshot with the same `INSERT ... SELECT`. Their visible DDL differs only because `m08_events_indexed` declares `m08_product_idx`; neither table has a materialized view.

The one-row result identifies the product used by both lookup queries and its matching event count. Choosing the least frequent product with positive revenue produces a selective predicate without hard-coding a data-dependent `product_id`. The copied data includes the single owned purchase appended in Section 3. Physical pages, compaction state, and cache residency need not be identical across two independently loaded tables.

In [ ]:
lookup_template = """
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM {table}
WHERE product_id = {product_id}
ORDER BY event_time, event_id, user_id
"""
unindexed_lookup = lab.capture_query(lookup_template.format(
    table="m08_events_unindexed", product_id=rare_product_id))
unindexed_lookup.show("Product lookup without the secondary index", detail=True)
indexed_lookup = lab.capture_query(lookup_template.format(
    table="m08_events_indexed", product_id=rare_product_id))
indexed_lookup.show("Product lookup with the inverted index", detail=True)
lab.assert_same_rows(unindexed_lookup.rows, indexed_lookup.rows)
assert len(indexed_lookup.rows) == int(rare_product.iloc[0]["matching_events"])
lab.assert_scan(unindexed_lookup, "m08_events_unindexed")
lab.assert_scan(indexed_lookup, "m08_events_indexed")
card("Every returned event matches. Read the per-task index counters before attributing scan savings.",
     "ok", "Selective lookup checked");

**Expected result:** the two queries return the same events. The scan label in `EXPLAIN` still names the table's base index; it does not by itself establish secondary-index filtering. Open the **DetailProfile scan counters** for each query and retain the Fragment, host, PipelineTask, and scan identity.

| Evidence | Interpretation |
|---|---|
| `RowsInvertedIndexFiltered` | Rows filtered through the inverted-index path in that scan task; positive values support actual index filtering. |
| `InvertedIndexQueryTime`, `InvertedIndexFilterTime` | Recorded work on the index path, with the original time units. They are not end-to-end query latency. |
| `InvertedIndexQueryCacheHit` / `Miss` | Index-cache events; the capture helper has not disabled this cache. |
| `RowsStatsFiltered`, `RowsKeyRangeFiltered` | Other pruning paths that may also reduce work. Do not attribute every skipped row to the new index. |
| `ScanRows` | Rows read from storage, including page-cache reads, under the named scan. |
| `RowsRead`, `RowsProduced` | Scanner/output row counters at their respective scopes; the final SQL result remains the correctness check. |
| `ScanBytes` | Recorded scan bytes, not a measurement of physical disk traffic avoided. |

On the tested fixture, the indexed run records positive `RowsInvertedIndexFiltered` values and fewer `ScanRows`. Exact counts and timings can change with storage layout and caches. If a counter is absent, it is **not shown**, not zero; use the complete raw Profile to inspect the evidence. The merged summary and the per-task detail describe overlapping work—never add them together.

A selective index trades additional storage and write/maintenance work for less lookup work. This comparison neither quantifies that tradeoff under production load nor proves a fixed speedup. No multi-BE performance conclusion follows from this single-node lab.

### What you have verified

| Need | Observable check |
|---|---|
| Reuse the purchase definition | Equal grouped results; ordinary-view plan still reads the base table. |
| Maintain a supported summary on write | New build finishes, the materialized index is selected, and one known purchase changes exactly one group. |
| Reuse multi-table computation with refresh lag | Direct stored data stays unchanged after reclassification, then matches after a new successful refresh task. |
| Match the question to stored grain and measures | Additive category totals use the asynchronous view; distinct buyers still need source detail. |
| Investigate a rare product | Identical input rows and answers, with scan-scoped runtime index evidence. |

The owned objects remain in `doris_course` for inspection. Restart at initialization to rebuild them. Capture settings have been restored; the course environment remains running.

### References

- [Synchronous materialized views](https://doris.apache.org/docs/4.x/query-acceleration/materialized-view/sync-materialized-view/): supported definitions, initial build, write maintenance, and column-name restrictions.
- [Manage and query asynchronous materialized views](https://doris.apache.org/docs/4.x/query-acceleration/materialized-view/async-materialized-view/functions-and-demands/): manual refresh, task state, freshness, and query rewrite.
- [Inverted Index](https://doris.apache.org/docs/4.x/key-features/inverted-index/): predicates and index construction.
- [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/): runtime evidence.
- [OLAP scan counters in the tested build](https://github.com/apache/doris/blob/7126cf65d96/be/src/exec/operator/olap_scan_operator.cpp): `ScanRows`, index-filter counters, and their units.